# IT23183186 Model B - Hive frame detection with YOLO11n

Train the repository's YOLO11n detector on a prepared FAIRHiveFrames dataset, audit the split, and evaluate the selected checkpoint on a held-out test set. The seven classes describe comb/frame structures; this model does not diagnose disease.

Run cells in order. Dataset preparation and training results are not included. Preserve the original images and annotations, and document group-aware split provenance separately: automated duplicate checks cannot prove that hive/session groups are independent.

## 1. Configure the project, dataset, and run

Select the intended Python kernel first. If dependencies are missing, run the printed installation command in a terminal, then restart the kernel. The adjacent requirements file installs **CPU-only** PyTorch. For CUDA training, install a compatible GPU-enabled PyTorch build separately before installing Ultralytics.

Set `PREPARED_DATASET_OVERRIDE` to an existing prepared directory containing `dataset.yaml`, or conventional `images/{train,val,test}` and matching `labels/{train,val,test}` folders. You can also set `HIVESENSE_FAIRHIVE_PREPARED_DIR` before launching Jupyter. Relative overrides are resolved from the repository root. No dataset is downloaded or fabricated by this notebook.

Keep the test split out of model selection. Edit the settings below before running later cells. Rerunning this cell starts a new run; use `EXISTING_RUN_NAME` to inspect a completed run without retraining.

In [1]:
from datetime import datetime
import json
import os
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path
import subprocess
import sys

MODEL_RELATIVE_DIR = Path('backend') / 'models' / 'IT23183186_model_b_hive_frame_detector'

def find_repository_root(start=None):
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / MODEL_RELATIVE_DIR).is_dir():
            return candidate
        for child in candidate.glob('*/backend/models/IT23183186_model_b_hive_frame_detector'):
            return child.parents[2]
    raise FileNotFoundError(
        f'Could not find the repository above {start}. Start Jupyter inside the project folder.'
    )

REPO_ROOT = find_repository_root()
MODEL_DIR = REPO_ROOT / MODEL_RELATIVE_DIR

# Set this to an EXISTING prepared copy, or set HIVESENSE_FAIRHIVE_PREPARED_DIR.
# This is not the ZIP archive or the original unsplit data/obj folder.
PREPARED_DATASET_OVERRIDE = None  # Example: Path(r'D:/bee-data/FAIRHiveFrames_prepared')
configured_root = PREPARED_DATASET_OVERRIDE or os.getenv('HIVESENSE_FAIRHIVE_PREPARED_DIR')
research_data = REPO_ROOT.parent.parent / 'DATAESET'
DATASET_CANDIDATES = [
    research_data / 'FAIRHiveFrames_prepared',
    research_data / 'FAIRHiveFrames-1K_prepared',
    REPO_ROOT / 'datasets' / 'FAIRHiveFrames_prepared',
    Path('C:/datasets/FAIRHiveFrames_prepared'),
]
def looks_prepared(path):
    return path.is_dir() and (
        (path / 'dataset.yaml').is_file() or
        all((path / 'images' / split).is_dir() and (path / 'labels' / split).is_dir()
            for split in ('train', 'val', 'test'))
    )
if configured_root:
    DATASET_ROOT = Path(configured_root).expanduser()
    if not DATASET_ROOT.is_absolute():
        DATASET_ROOT = REPO_ROOT / DATASET_ROOT
    DATASET_ROOT = DATASET_ROOT.resolve()
else:
    available_roots = [path for path in DATASET_CANDIDATES if looks_prepared(path)]
    if len(available_roots) > 1:
        raise RuntimeError('Multiple prepared copies found. Set PREPARED_DATASET_OVERRIDE explicitly: ' + str(available_roots))
    DATASET_ROOT = available_roots[0] if available_roots else None
DATA_YAML = DATASET_ROOT / 'dataset.yaml' if DATASET_ROOT else None
RUN_ID = datetime.now().strftime('IT23183186_ModelB_%Y%m%d_%H%M%S_%f')
RUNS_DIR = MODEL_DIR / 'runs'
AUDIT_REPORT = MODEL_DIR / 'reports' / f'dataset_audit_{RUN_ID}.json'
EXISTING_RUN_NAME = None  # A completed folder name under runs/, for results only.
RUN_NAME = EXISTING_RUN_NAME or f'{RUN_ID}_YOLO11n'
if Path(RUN_NAME).name != RUN_NAME or RUN_NAME in ('.', '..'):
    raise ValueError('Run name must be a single directory name.')
EVALUATION_NAME = f'{RUN_NAME}_test'
PYTHON = sys.executable

# Shared configuration used by training and final test evaluation.
EPOCHS = 100
IMAGE_SIZE = 640
BATCH_SIZE = 8
PATIENCE = 20
WORKERS = 0 if os.name == 'nt' else 4
SEED = 2026
DEVICE = 'auto'  # 'auto', 'cpu', or '0' for the first CUDA GPU

# Import this project's class map without relying on a cached generic `model` module.
import importlib.util
spec = importlib.util.spec_from_file_location('hivesense_model_b', MODEL_DIR / 'model.py')
model_b = importlib.util.module_from_spec(spec)
spec.loader.exec_module(model_b)
CLASS_NAMES = model_b.FAIRHIVE_CLASS_NAMES

def require_dataset():
    if DATASET_ROOT is None or not DATASET_ROOT.is_dir():
        raise FileNotFoundError(
            'Prepared FAIRHiveFrames data not found. Set PREPARED_DATASET_OVERRIDE '
            'to your existing prepared directory, then rerun configuration. See the adjacent README.md.'
        )
    if DATA_YAML is None or not DATA_YAML.is_file():
        raise FileNotFoundError('Run the dataset YAML cell before validation or training.')

def runtime_device():
    try:
        import torch
        import ultralytics
    except ImportError as exc:
        raise RuntimeError('Install dependencies in this kernel and restart it. See the setup instructions.') from exc
    selected = ('0' if torch.cuda.is_available() else 'cpu') if DEVICE == 'auto' else DEVICE
    if selected != 'cpu' and not torch.cuda.is_available():
        raise RuntimeError('CUDA requested but unavailable in this kernel. Choose cpu or install a CUDA-enabled PyTorch build.')
    print(f'PyTorch {torch.__version__}; Ultralytics {ultralytics.__version__}; device={selected}')
    return selected

def run(args, *, check=True):
    if any(value is None for value in args):
        raise ValueError('A command argument is missing. Run the configuration and dataset YAML cells first.')
    command = [str(value) for value in args]
    print('Running:', subprocess.list2cmdline(command), flush=True)
    # Pipe and forward output so progress is visible in VS Code/Jupyter on Windows.
    with subprocess.Popen(command, cwd=REPO_ROOT, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, encoding='utf-8', errors='replace',
                          env={**os.environ, 'PYTHONUNBUFFERED': '1', 'PYTHONIOENCODING': 'utf-8'}) as process:
        try:
            for line in process.stdout:
                print(line, end='', flush=True)
            returncode = process.wait()
        except KeyboardInterrupt:
            process.terminate()
            process.wait()
            raise
    if check and returncode:
        raise subprocess.CalledProcessError(returncode, command)
    return returncode

print('Repository root:', REPO_ROOT)
print('Model scripts:', MODEL_DIR)
print('Prepared dataset:', DATASET_ROOT or 'NOT FOUND - training is not ready')
if DATASET_ROOT is None:
    print('Checked locations:')
    for candidate in DATASET_CANDIDATES:
        print(' -', candidate)
    print('Locate the official FAIRHiveFrames images and annotations, prepare verified splits, then set PREPARED_DATASET_OVERRIDE and rerun this cell.')
try:
    print('Ultralytics version:', version('ultralytics'))
except PackageNotFoundError:
    print('Ultralytics is not installed in this notebook kernel; install the requirements above.')

print('Kernel:', PYTHON)
print('CPU installation command:', subprocess.list2cmdline([PYTHON, '-m', 'pip', 'install', '-r', str(MODEL_DIR / 'requirements.txt')]))
print('Run:', RUN_NAME)

Repository root: C:\Users\User\OneDrive\Desktop\Y4S1 NOTES\Reserach\PROJECT_V1\J26-IT-378-Research-Project
Model scripts: C:\Users\User\OneDrive\Desktop\Y4S1 NOTES\Reserach\PROJECT_V1\J26-IT-378-Research-Project\backend\models\IT23183186_model_b_hive_frame_detector
Prepared dataset: NOT FOUND - training is not ready
Checked locations:
 - C:\Users\User\OneDrive\Desktop\Y4S1 NOTES\Reserach\DATAESET\FAIRHiveFrames_prepared
 - C:\Users\User\OneDrive\Desktop\Y4S1 NOTES\Reserach\DATAESET\FAIRHiveFrames-1K_prepared
 - C:\Users\User\OneDrive\Desktop\Y4S1 NOTES\Reserach\PROJECT_V1\J26-IT-378-Research-Project\datasets\FAIRHiveFrames_prepared
 - C:\datasets\FAIRHiveFrames_prepared
Locate the official FAIRHiveFrames images and annotations, prepare verified splits, then set PREPARED_DATASET_OVERRIDE and rerun this cell.
Ultralytics is not installed in this notebook kernel; install the requirements above.
Kernel: c:\Users\User\anaconda3\python.exe
CPU installation command: c:\Users\User\anaconda3\py

## 2. Create or check the dataset YAML

If `dataset.yaml` is absent, the cell creates it inside the prepared dataset folder using the conventional split folders and the fixed class map. This creates configuration only; it does not create or change any annotations. If a YAML already exists, it is preserved and checked. Make sure its paths refer to the actual prepared split folders.

In [2]:
if DATASET_ROOT is None or not DATASET_ROOT.is_dir():
    raise FileNotFoundError('Set PREPARED_DATASET_OVERRIDE to an existing prepared dataset and rerun configuration. See README.md.')

if not DATA_YAML.exists():
    import yaml
    required_dirs = [DATASET_ROOT / kind / split for kind in ('images', 'labels') for split in ('train', 'val', 'test')]
    missing = [str(path) for path in required_dirs if not path.is_dir()]
    if missing:
        raise FileNotFoundError('Expected images/train, images/val, and images/test. Missing: ' + ', '.join(missing))
    config = {
        'path': DATASET_ROOT.resolve().as_posix(),
        'train': 'images/train',
        'val': 'images/val',
        'test': 'images/test',
        'names': {index: name for index, name in enumerate(CLASS_NAMES)},
    }
    DATA_YAML.write_text(yaml.safe_dump(config, sort_keys=False), encoding='utf-8')
    print('Created dataset config:', DATA_YAML)
else:
    print('Using existing dataset config:', DATA_YAML)

config, resolved_root = model_b.read_dataset_yaml(DATA_YAML, require_test=True)
# Normalize the root in a run-specific copy so Ultralytics uses the same paths as the audit.
# Preserve the user's source YAML.
import yaml
source_yaml = DATA_YAML
config['path'] = resolved_root.as_posix()
DATA_YAML = MODEL_DIR / 'reports' / RUN_ID / 'dataset.resolved.yaml'
DATA_YAML.parent.mkdir(parents=True, exist_ok=True)
DATA_YAML.write_text(yaml.safe_dump(config, sort_keys=False), encoding='utf-8')
print('Source YAML:', source_yaml)
print('Run YAML:', DATA_YAML)
print('Resolved dataset root:', resolved_root)
print('Splits:', {name: config[name] for name in ('train', 'val', 'test')})
print('Verified class names:', config['names'])

FileNotFoundError: Set PREPARED_DATASET_OVERRIDE to an existing prepared dataset and rerun configuration. See README.md.

## 3. Validate images, labels, boxes, duplicates, and split separation

This check must pass before training. It checks image/label matches, readable images, YOLO box format and bounds, class IDs, exact duplicates, near-duplicate candidates, and detectable group overlap. Review the saved JSON report as well. If it fails or flags near-duplicate candidates, resolve the dataset issue without inventing labels, then rerun validation.

In [ ]:
require_dataset()

def validate_prepared_dataset():
    returncode = run([
        PYTHON, MODEL_DIR / 'validate_dataset.py',
        '--data', DATA_YAML, '--output', AUDIT_REPORT,
    ], check=False)
    if not AUDIT_REPORT.is_file():
        raise RuntimeError('Validator did not create a report. Review the command output above.')
    audit = json.loads(AUDIT_REPORT.read_text(encoding='utf-8'))
    print('Audit report:', AUDIT_REPORT)
    for split, summary in audit.get('splits', {}).items():
        print(f"{split}: {summary['image_count']} images, {summary['annotation_count']} boxes")
    empty_splits = [split for split in ('train', 'val', 'test')
                    if audit.get('splits', {}).get(split, {}).get('image_count', 0) == 0]
    if returncode or audit.get('passed') is not True or empty_splits:
        raise RuntimeError(f'Dataset audit failed or contains empty splits {empty_splits}. Review {AUDIT_REPORT}')
    return audit

audit = validate_prepared_dataset()
print('Dataset validation passed. Confirm split provenance before training.')

## 4. Train YOLO11n

Run after the dataset audit passes. Training uses the training split and validation split for early stopping and checkpoint selection. It leaves the test split for the final evaluation cell. The unique run name prevents an earlier run from being overwritten. Default settings are 100 epochs maximum, 640 image size, batch 8, patience 20, and seed 2026; adjust batch size if memory is limited.

In [ ]:
require_dataset()
if EXISTING_RUN_NAME:
    raise RuntimeError('EXISTING_RUN_NAME is for inspecting completed runs. Set it to None for new training.')
if (RUNS_DIR / RUN_NAME).exists():
    raise FileExistsError('This run already exists. Rerun configuration for a new run name; existing results are preserved.')
selected_device = runtime_device()
# Recheck the actual dataset immediately before training; do not trust a stale report.
audit = validate_prepared_dataset()
run([
    PYTHON, MODEL_DIR / 'train.py', '--data', DATA_YAML,
    '--epochs', EPOCHS, '--imgsz', IMAGE_SIZE, '--batch', BATCH_SIZE,
    '--patience', PATIENCE, '--workers', WORKERS, '--seed', SEED,
    '--device', selected_device, '--project', RUNS_DIR, '--name', RUN_NAME,
])
BEST_WEIGHTS = RUNS_DIR / RUN_NAME / 'weights' / 'best.pt'
if not BEST_WEIGHTS.is_file():
    raise FileNotFoundError(f'Training returned without a best checkpoint: {BEST_WEIGHTS}')
print('Best checkpoint:', BEST_WEIGHTS)

## 5. Evaluate once on the held-out test split

Run only after training finishes and `best.pt` exists. The evaluation script selects `split=test` and writes actual metrics and plots to a separate run directory. These measurements describe detection of the named hive structures; they do not indicate Varroa, AFB, EFB, or any other disease.

In [ ]:
require_dataset()
BEST_WEIGHTS = RUNS_DIR / RUN_NAME / 'weights' / 'best.pt'
if not BEST_WEIGHTS.is_file():
    raise FileNotFoundError(f'Best checkpoint not found: {BEST_WEIGHTS}. Complete training first.')
if (RUNS_DIR / EVALUATION_NAME).exists():
    raise FileExistsError('Test output already exists. Inspect the saved results below.')
run([
    PYTHON, MODEL_DIR / 'evaluate.py', '--data', DATA_YAML,
    '--weights', BEST_WEIGHTS, '--imgsz', IMAGE_SIZE, '--batch', BATCH_SIZE,
    '--device', runtime_device(), '--project', RUNS_DIR, '--name', EVALUATION_NAME,
])
TEST_REPORT = RUNS_DIR / EVALUATION_NAME / 'test_evaluation.json'
if not TEST_REPORT.is_file():
    raise FileNotFoundError(f'Evaluation did not produce its report: {TEST_REPORT}')
print('Evaluation report:', TEST_REPORT)

## Saved artifacts

- Dataset audit: `reports/dataset_audit_<run>.json`
- Resolved dataset configuration: `reports/<run>/dataset.resolved.yaml`
- Best checkpoint and training history: `runs/<run>/weights/best.pt` and `runs/<run>/results.csv`
- Final test metrics and plots: `runs/<run>_test/test_evaluation.json` and adjacent PNG files

All paths above are relative to this notebook's model directory. Keep the checkpoint, configuration, environment versions, audit, and split provenance together for research reporting.

References: [Ultralytics YOLO11](https://docs.ultralytics.com/models/yolo11/), [training settings](https://docs.ultralytics.com/modes/train/), and [validation](https://docs.ultralytics.com/modes/val/).

## Display saved evaluation graphs
Run after successful training and evaluation. Shows real Ultralytics outputs if present. Missing outputs mean evaluation is not available, not zero performance.


In [ ]:
import json
from IPython.display import Image, display, Markdown
report = RUNS_DIR / EVALUATION_NAME / 'test_evaluation.json'
if report.is_file():
    print(json.dumps(json.loads(report.read_text(encoding='utf-8')), indent=2))
else:
    print('No final Model B test evaluation found. Complete dataset validation, training and evaluation first.')
for folder in (RUNS_DIR / RUN_NAME, RUNS_DIR / EVALUATION_NAME):
    for name in ('results.png', 'confusion_matrix.png', 'confusion_matrix_normalized.png', 'BoxPR_curve.png', 'BoxF1_curve.png', 'BoxP_curve.png', 'BoxR_curve.png'):
        graph = folder / name
        if graph.is_file():
            display(Markdown(f'**{folder.name} / {name}**'))
            display(Image(filename=str(graph), width=1000))